In [5]:
# If needed, uncomment to install Open3D in the current environment
# !pip install open3d

import numpy as np
import open3d as o3d
import plotly.graph_objects as go
from pathlib import Path

# Update these paths to your .ply files
base_ply_path = Path("results/board_slow/focus_edges.npy")
edge_ply_path = Path("results/board_slow//pointcloud_edges.ply")

# What to show: 'base' | 'edges' | 'both'
show = "both"

# Optional: downsample for faster rendering (set to 0 to disable)
voxel_downsample = 0.0

# Optional: random subsample cap (Plotly will struggle with millions of points)
max_points = 10000

# --- Filtering ---
# Option A: remove points with any coordinate |x|,|y|,|z| > max_abs_coord
max_abs_coord = 20

# Option B: remove points with distance from origin > max_radius
max_radius = 20

# Choose which filter to apply: 'abs' or 'radius'
filter_mode = "radius"

def _load_pcd(path: Path):
    if not path.exists():
        print(f"Missing: {path}")
        return None

    suf = path.suffix.lower()

    # --- NPY: assume Nx3 (or Nx>=3; we take first 3 as xyz) ---
    if suf == ".npy":
        arr = np.load(str(path))
        arr = np.asarray(arr)

        # handle shapes like (1, N, 3) or (N, 3)
        arr = arr.reshape(-1, arr.shape[-1]) if arr.ndim > 2 else arr
        if arr.shape[1] < 3:
            raise ValueError(f"{path}: expected at least 3 columns (xyz), got shape {arr.shape}")

        xyz = arr[:, :3].astype(np.float64)

        # remove NaN/Inf
        mask = np.isfinite(xyz).all(axis=1)
        xyz = xyz[mask]

        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(xyz)
        print(path, pcd, "(loaded from npy)")
        return pcd

    # --- NPZ (optional): if you ever have xyz stored inside a key ---
    if suf == ".npz":
        data = np.load(str(path))
        # try common keys
        for key in ["points", "xyz", "cloud", "verts"]:
            if key in data:
                xyz = np.asarray(data[key]).reshape(-1, 3).astype(np.float64)
                pcd = o3d.geometry.PointCloud()
                pcd.points = o3d.utility.Vector3dVector(xyz)
                print(path, pcd, f"(loaded from npz key='{key}')")
                return pcd
        raise ValueError(f"{path}: npz has keys {list(data.keys())}, none match expected ones")

    # --- PLY/PCD/etc: Open3D handles these ---
    pcd = o3d.io.read_point_cloud(str(path))
    print(path, pcd, "(loaded from file)")
    return pcd


def _filter_pcd(pcd: o3d.geometry.PointCloud):
    pts = np.asarray(pcd.points)
    if pts.size == 0:
        return pcd
    mask_abs = (np.abs(pts) <= max_abs_coord).all(axis=1)
    mask_rad = np.linalg.norm(pts, axis=1) <= max_radius
    mask = mask_rad if filter_mode == "radius" else mask_abs
    return pcd.select_by_index(np.where(mask)[0])

def _maybe_downsample(pcd: o3d.geometry.PointCloud):
    if voxel_downsample and voxel_downsample > 0:
        return pcd.voxel_down_sample(voxel_size=float(voxel_downsample))
    return pcd

def _maybe_subsample(pcd: o3d.geometry.PointCloud):
    if not max_points or max_points <= 0:
        return pcd
    n = len(pcd.points)
    if n <= max_points:
        return pcd
    idx = np.random.choice(n, size=int(max_points), replace=False)
    return pcd.select_by_index(idx)

pcd_base = _load_pcd(base_ply_path)
pcd_edge = _load_pcd(edge_ply_path)

if pcd_base is None and pcd_edge is None:
    raise FileNotFoundError("No point clouds found. Check base_ply_path / edge_ply_path.")

geoms = []
if show in ("base", "both") and pcd_base is not None:
    p = _maybe_subsample(_maybe_downsample(_filter_pcd(pcd_base)))
    p.paint_uniform_color([0.75, 0.75, 0.75])
    geoms.append(p)

if show in ("edges", "both") and pcd_edge is not None:
    p = _maybe_subsample(_maybe_downsample(_filter_pcd(pcd_edge)))
    p.paint_uniform_color([1.0, 0.2, 0.2])
    geoms.append(p)

if not geoms:
    raise RuntimeError(f"Nothing to show (show={show}).")

def _scatter_from_pcd(pcd: o3d.geometry.PointCloud, name: str, color: str, size: int = 1):
    pts = np.asarray(pcd.points)
    if pts.size == 0:
        return None
    return go.Scatter3d(
        x=pts[:, 0],
        y=pts[:, 1],
        z=pts[:, 2],
        mode="markers",
        name=name,
        marker=dict(size=size, color=color, opacity=0.85),
    )

traces = []
if show in ("base", "both") and pcd_base is not None:
    pb = _maybe_subsample(_maybe_downsample(_filter_pcd(pcd_base)))
    print(f"Base points shown: {len(pb.points)}")
    t = _scatter_from_pcd(pb, name="base", color="rgb(0,0,255)", size=1)
    if t is not None:
        traces.append(t)

if show in ("edges", "both") and pcd_edge is not None:
    pe = _maybe_subsample(_maybe_downsample(_filter_pcd(pcd_edge)))
    print(f"Edge points shown: {len(pe.points)}")
    t = _scatter_from_pcd(pe, name="edges", color="rgb(255,60,60)", size=1)
    if t is not None:
        traces.append(t)

if not traces:
    raise RuntimeError("No points to plot after filtering.")

fig = go.Figure(data=traces)
fig.update_layout(
    scene=dict(aspectmode="data"),
    margin=dict(l=0, r=0, t=30, b=0),
    title=f"Point Cloud Compare (show={show})",
)
fig.show()


results/board_slow/focus_edges.npy PointCloud with 1328 points. (loaded from npy)
results/board_slow/pointcloud_edges.ply PointCloud with 920581 points. (loaded from file)
Base points shown: 1328
Edge points shown: 10000
